# 2. Experiment Analysis: Hypothesis Testing

This notebook tests whether the advertising treatment causes incremental conversions/visits.

## Methodology
- **Hypothesis:** Treatment group has higher conversion rate than control
- **Test:** Two-sample z-test for proportions
- **Significance level:** α = 0.05 (95% confidence)
- **Effect size:** Measured as absolute and relative lift

## Analyses
1. Conversion rate comparison (z-test + 95% CI)
2. Visit rate comparison (z-test + 95% CI)
3. Statistical power analysis
4. Minimum detectable effect (MDE) at 80% power

## Output
- P-values indicating statistical significance
- 95% confidence intervals on the treatment effect
- Power metrics for design decisions

In [1]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize
)

from statsmodels.stats.power import NormalIndPower
from scipy.optimize import brentq

In [ ]:
import os
DATA_PATH = os.path.join("data", "raw", "criteo-uplift-v2.1.csv.gz")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (13979592, 16)


In [3]:
group_metrics = (
    df.groupby("treatment")
      .agg(
          observations=("treatment", "size"),
          conversions=("conversion", "sum"),
          visits=("visit", "sum"),
          exposures=("exposure", "sum")
      )
      .reset_index()
)

group_metrics["conversion_rate"] = (
    group_metrics["conversions"] / group_metrics["observations"]
)

group_metrics["visit_rate"] = (
    group_metrics["visits"] / group_metrics["observations"]
)

group_metrics

,treatment,observations,conversions,visits,exposures,conversion_rate,visit_rate
0,0,2096937,4063,80105,0,0.001938,0.038201
1,1,11882655,36711,576824,428212,0.003089,0.048543


In [4]:
control = group_metrics[group_metrics["treatment"] == 0].iloc[0]
treatment = group_metrics[group_metrics["treatment"] == 1].iloc[0]

# Conversion
conversion_absolute_lift = (
    treatment["conversion_rate"] - control["conversion_rate"]
)

conversion_relative_lift = (
    conversion_absolute_lift / control["conversion_rate"]
)

# Visit
visit_absolute_lift = (
    treatment["visit_rate"] - control["visit_rate"]
)

visit_relative_lift = (
    visit_absolute_lift / control["visit_rate"]
)

print("CONVERSION")
print(f"Control rate:          {control['conversion_rate']:.6%}")
print(f"Treatment rate:        {treatment['conversion_rate']:.6%}")
print(f"Absolute lift:         {conversion_absolute_lift:.6%}")
print(f"Relative lift:         {conversion_relative_lift:.2%}")

print("\nVISIT")
print(f"Control rate:          {control['visit_rate']:.6%}")
print(f"Treatment rate:        {treatment['visit_rate']:.6%}")
print(f"Absolute lift:         {visit_absolute_lift:.6%}")
print(f"Relative lift:         {visit_relative_lift:.2%}")

CONVERSION
Control rate:          0.193759%
Treatment rate:        0.308946%
Absolute lift:         0.115187%
Relative lift:         59.45%

VISIT
Control rate:          3.820096%
Treatment rate:        4.854336%
Absolute lift:         1.034240%
Relative lift:         27.07%


In [5]:
from statsmodels.stats.proportion import proportions_ztest

conversion_counts = [
    int(control["conversions"]),
    int(treatment["conversions"])
]

conversion_nobs = [
    int(control["observations"]),
    int(treatment["observations"])
]

z_stat, p_value = proportions_ztest(
    conversion_counts,
    conversion_nobs
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value:     {p_value:.10f}")

Z-statistic: -28.5165
P-value:     0.0000000000


In [6]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=int(treatment["conversions"]),
    nobs1=int(treatment["observations"]),
    count2=int(control["conversions"]),
    nobs2=int(control["observations"]),
    method="wald"
)

print(f"95% CI lower: {ci_low:.6%}")
print(f"95% CI upper: {ci_high:.6%}")

95% CI lower: 0.108451%
95% CI upper: 0.121924%


In [7]:
visit_counts = [
    int(control["visits"]),
    int(treatment["visits"])
]

visit_nobs = [
    int(control["observations"]),
    int(treatment["observations"])
]

visit_z_stat, visit_p_value = proportions_ztest(
    visit_counts,
    visit_nobs
)

print(f"Z-statistic: {visit_z_stat:.4f}")
print(f"P-value:     {visit_p_value:.10f}")

Z-statistic: -65.2474
P-value:     0.0000000000


In [8]:
visit_ci_low, visit_ci_high = confint_proportions_2indep(
    count1=int(treatment["visits"]),
    nobs1=int(treatment["observations"]),
    count2=int(control["visits"]),
    nobs2=int(control["observations"]),
    method="wald"
)

print(f"95% CI lower: {visit_ci_low:.6%}")
print(f"95% CI upper: {visit_ci_high:.6%}")

95% CI lower: 1.005563%
95% CI upper: 1.062918%


In [9]:
expected_control_conversions = (
    treatment["observations"] * control["conversion_rate"]
)

incremental_conversions = (
    treatment["conversions"] - expected_control_conversions
)

print(f"Expected conversions at control rate: {expected_control_conversions:,.0f}")
print(f"Observed treatment conversions:       {treatment['conversions']:,.0f}")
print(f"Estimated incremental conversions:     {incremental_conversions:,.0f}")

Expected conversions at control rate: 23,024
Observed treatment conversions:       36,711
Estimated incremental conversions:     13,687


In [10]:
baseline_rate = control["conversion_rate"]
treatment_rate = treatment["conversion_rate"]

incremental_conversions_per_million = (
    (treatment_rate - baseline_rate) * 1_000_000
)

print(
    f"Estimated incremental conversions per 1M treated users: "
    f"{incremental_conversions_per_million:,.0f}"
)

Estimated incremental conversions per 1M treated users: 1,152


In [11]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

alpha = 0.05

baseline_conversion = control["conversion_rate"]
observed_conversion = treatment["conversion_rate"]

effect_size = proportion_effectsize(
    baseline_conversion,
    observed_conversion
)

power_analysis = NormalIndPower()

observed_power = power_analysis.solve_power(
    effect_size=effect_size,
    nobs1=int(control["observations"]),
    alpha=alpha,
    ratio=treatment["observations"] / control["observations"],
    alternative="two-sided"
)

print(f"Baseline conversion rate: {baseline_conversion:.6%}")
print(f"Observed treatment rate:  {observed_conversion:.6%}")
print(f"Effect size:              {effect_size:.6f}")
print(f"Observed statistical power: {observed_power:.4%}")

Baseline conversion rate: 0.193759%
Observed treatment rate:  0.308946%
Effect size:              -0.023159
Observed statistical power: 100.0000%


In [12]:
target_power = 0.80

mde_effect_size = power_analysis.solve_power(
    effect_size=None,
    nobs1=int(control["observations"]),
    alpha=alpha,
    power=target_power,
    ratio=treatment["observations"] / control["observations"],
    alternative="two-sided"
)

print(f"MDE standardized effect size: {mde_effect_size:.6f}")

MDE standardized effect size: 0.002098


In [14]:
from scipy.optimize import brentq
from statsmodels.stats.proportion import proportion_effectsize

mde_standardized = abs(mde_effect_size)
baseline = baseline_conversion

def effect_difference(p2):
    return abs(proportion_effectsize(baseline, p2)) - mde_standardized

mde_treatment_rate = brentq(
    effect_difference,
    baseline + 1e-10,
    0.01
)

mde_absolute_lift = mde_treatment_rate - baseline
mde_relative_lift = mde_absolute_lift / baseline

print(f"Baseline conversion rate: {baseline:.6%}")
print(f"MDE treatment rate:       {mde_treatment_rate:.6%}")
print(f"MDE absolute lift:        {mde_absolute_lift:.6%}")
print(f"MDE relative lift:        {mde_relative_lift:.2%}")

Baseline conversion rate: 0.193759%
MDE treatment rate:       0.203096%
MDE absolute lift:        0.009338%
MDE relative lift:        4.82%


In [ ]:
# Save main results
results = pd.DataFrame({
    'Metric': ['Conversion Absolute Lift', 'Conversion Relative Lift', 'Visit Absolute Lift', 'Visit Relative Lift'],
    'Value': [conversion_absolute_lift, conversion_relative_lift, visit_absolute_lift, visit_relative_lift]
})
results.to_csv('outputs/results/experiment_summary.csv', index=False)
print("✅ Results saved to outputs/results/experiment_summary.csv")